# 18.5 如何顯示較低機率的候選？


沿用同一題和候選 `[4,3,100]`，這次改用更尖的人工分數 `[4,1,0]`，讓第一名占約94%。其他候選只剩少許份量，這些細小比例在軟目標誤差中的加權份量很小。可以讓分布平一些，保留排名但縮小差距嗎？沿用[softmax尺度](https://birdhackor.github.io/tiny-perceptron-vlm/14.3.html)，先把教師logits除以正數T，再softmax；這個T叫蒸餾temperature（溫度），T越大通常分布越平。

取候選 `[4,3,100]`的分數 `[4,1,0]`。`exp(x)`就是`e ** x`，e約為2.718，把分數轉為正數份量。softmax把各份量除以同一個總和；兩個機率相除時，共同總和消掉，所以候選4比候選3是`exp(4/T) / exp(1/T) = exp(3/T)`。T=1時，第一和第二分數差3，機率比為 `exp(3)`約20.1；T=2時，差縮為1.5，比例約4.48，所以第二名相對份量增加。T=4時差0.75，比例約2.12。沒有改教師排名，改的是它展示給學生的選項差距；溫度不是答案正確率，也不是讓教師重新推理一次。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

logits = torch.tensor([4.0, 1.0, 0.0])
for temperature in (1.0, 2.0, 4.0):
    probability = (logits / temperature).softmax(0)
    print(
        "T",
        temperature,
        "機率",
        probability.round(decimals=4).tolist(),
        "第一比第二",
        round((probability[0] / probability[1]).item(), 4),
    )

輸入三個固定分數，不用已訓練教師。`softmax(0)`沿候選軸轉成機率；temperature在softmax前除，不是將最後機率隨便除T，那會讓總和不再為一。T=1機率約 `[0.9362,0.0466,0.0171]`，T=2約 `[0.7361,0.1643,0.0996]`，T=4約 `[0.5434,0.2567,0.1999]`。最後一個候選從1.71%增到19.99%，因此比較能參與學生的學習誤差。

下圖每條的完整長度都是100%機率，由左到右的顏色固定對應候選4、3、100。溫度提高時，橘色與綠色份量變大，藍色候選4仍是第一名；這表示次要選項更清楚，不表示答案排名改了。百分比四捨五入到小數點後兩位，條寬按原始機率繪製。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-18-temperature.svg")))

分布更平不一定更有用。T非常大時各候選接近平均，原本有意義的差異也被抹掉；教師給錯候選的份量同樣會變得更可見。需要用獨立驗證集選T：留出沒有用來訓練學生的題目，比較不同T訓練出的學生在這批題目上的回答品質，再決定T，而不是只希望所有候選都平均。最後報成績的測試題另留，不拿來反覆選T；資料分工可回看[1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html)。對學生也應使用相同T來形成比較分布，否則比較的是兩套不同尺度。

除以T也會改變分數對誤差的敏感度。本教材[分布蒸餾誤差](https://birdhackor.github.io/tiny-perceptron-vlm/18.8.html)會使用T²乘數作為尺度約定，並明確記在公式與函數中；這不是T讓教師突然多教T²倍知識。使用計算蒸餾誤差的輔助函式時，需知道是否已內建乘數，避免重複相乘；例如18.8的`distillation_kl`負責比較兩邊分布，內部已乘T²，外層就不用再乘。

蒸餾T和[生成抽樣設定](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html)的溫度應分開記錄。這裡改的是訓練用的教師/學生分布，不是要求最終回答以相同T抽樣；同一個蒸餾學生可以使用不同生成策略。

練習只把logits改成 `[8,2,0]`，temperature列表不變。先預測新T=2與原T=1結果相同，新T=4與原T=2相同，再執行核對。兩者都是相同的「分數除溫度」，這能幫你分辨絕對數值與相對尺度的作用。

<details>
<summary>選讀：原論文、T²約定與本次設定</summary>

[Hinton等人的原文第2節](https://arxiv.org/abs/1503.02531v1)先讓教師與學生都用同一高溫softmax，再以T²調整軟目標梯度的相對尺度。本輪正式實驗預設T=2、真值與教師各占一半，報告逐支記錄`temperature_squared_applied_once=true`；沒有跑多個溫度挑最好成績，不能把T=2寫成這些任務的最佳設定。最終留出生成則用greedy，不沿用蒸餾溫度抽樣。

</details>
